In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # JobFlow AI — Transformação Silver
# MAGIC
# MAGIC Este notebook transforma os dados Bronze da RemoteOK em uma tabela Silver:
# MAGIC
# MAGIC - interpreta o JSON bruto
# MAGIC - limpa HTML da descrição da vaga
# MAGIC - normaliza campos principais
# MAGIC - deduplica vagas
# MAGIC - cria colunas úteis para busca, ranking e futura camada Gold

# COMMAND ----------

from datetime import datetime, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

# COMMAND ----------

CATALOG = "workspace"
SCHEMA = "jobflow_ai"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_remoteok_jobs"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_remoteok_jobs"

print("=" * 70)
print("JOBFLOW AI — TRANSFORMAÇÃO SILVER")
print("=" * 70)
print(f"Bronze table: {BRONZE_TABLE}")
print(f"Silver table: {SILVER_TABLE}")
print(f"Horário UTC: {datetime.now(timezone.utc).isoformat()}")
print("=" * 70)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Validar contexto e dados Bronze

# COMMAND ----------

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

bronze_df = spark.table(BRONZE_TABLE)

print(f"Registros totais na Bronze: {bronze_df.count()}")

display(
    bronze_df.groupBy("source")
    .agg(
        F.count("*").alias("records"),
        F.countDistinct("source_job_id").alias("distinct_jobs"),
        F.max("ingested_at").alias("last_ingested_at"),
    )
    .orderBy("source")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Definir schema do payload JSON

# COMMAND ----------

remoteok_payload_schema = T.StructType(
    [
        T.StructField("id", T.StringType(), True),
        T.StructField("slug", T.StringType(), True),
        T.StructField("company", T.StringType(), True),
        T.StructField("position", T.StringType(), True),
        T.StructField("location", T.StringType(), True),
        T.StructField("date", T.StringType(), True),
        T.StructField("tags", T.ArrayType(T.StringType()), True),
        T.StructField("description", T.StringType(), True),
        T.StructField("salary_min", T.LongType(), True),
        T.StructField("salary_max", T.LongType(), True),
        T.StructField("apply_url", T.StringType(), True),
        T.StructField("url", T.StringType(), True),
    ]
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Parsear e limpar os dados

# COMMAND ----------

parsed_df = (
    bronze_df
    .withColumn("payload_struct", F.from_json(F.col("payload"), remoteok_payload_schema))
    .select(
        F.col("run_id"),
        F.col("source"),
        F.coalesce(F.col("source_job_id"), F.col("payload_struct.id")).alias("source_job_id"),
        F.coalesce(F.col("slug"), F.col("payload_struct.slug")).alias("slug"),
        F.coalesce(F.col("company"), F.col("payload_struct.company")).alias("company_raw"),
        F.coalesce(F.col("position"), F.col("payload_struct.position")).alias("position_raw"),
        F.coalesce(F.col("location"), F.col("payload_struct.location")).alias("location_raw"),
        F.coalesce(F.col("date"), F.col("payload_struct.date")).alias("published_at_raw"),
        F.coalesce(F.col("salary_min"), F.col("payload_struct.salary_min")).alias("salary_min"),
        F.coalesce(F.col("salary_max"), F.col("payload_struct.salary_max")).alias("salary_max"),
        F.coalesce(F.col("apply_url"), F.col("payload_struct.apply_url")).alias("apply_url"),
        F.coalesce(F.col("job_url"), F.col("payload_struct.url")).alias("job_url"),
        F.col("payload_struct.tags").alias("tags_array"),
        F.col("payload_struct.description").alias("description_html"),
        F.col("payload").alias("payload_json"),
        F.col("ingested_at"),
        F.col("file_path"),
    )
)

display(parsed_df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Normalizar texto, HTML, salário e localização

# COMMAND ----------

silver_base_df = (
    parsed_df
    .withColumn("company", F.trim(F.col("company_raw")))
    .withColumn("position", F.trim(F.col("position_raw")))
    .withColumn("location", F.trim(F.col("location_raw")))
    .withColumn(
        "description_text_step_1",
        F.regexp_replace(F.coalesce(F.col("description_html"), F.lit("")), "<[^>]+>", " ")
    )
    .withColumn(
        "description_text_step_2",
        F.regexp_replace(F.col("description_text_step_1"), "&nbsp;|&amp;|&quot;|&#39;", " ")
    )
    .withColumn(
        "description_text",
        F.trim(F.regexp_replace(F.col("description_text_step_2"), "\\s+", " "))
    )
    .withColumn("position_normalized", F.lower(F.col("position")))
    .withColumn("company_normalized", F.lower(F.col("company")))
    .withColumn("location_normalized", F.lower(F.col("location")))
    .withColumn(
        "tags_text",
        F.when(
            F.col("tags_array").isNotNull(),
            F.concat_ws(", ", F.col("tags_array"))
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "search_text",
        F.concat_ws(
            " ",
            F.col("position"),
            F.col("company"),
            F.col("location"),
            F.col("tags_text"),
            F.col("description_text"),
        )
    )
    .withColumn(
        "is_remote",
        F.when(
            F.lower(F.concat_ws(" ", F.col("location"), F.col("description_text"))).contains("remote"),
            F.lit(True),
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "has_salary",
        F.when(
            F.col("salary_min").isNotNull() | F.col("salary_max").isNotNull(),
            F.lit(True),
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "salary_midpoint",
        F.when(
            F.col("salary_min").isNotNull() & F.col("salary_max").isNotNull(),
            (F.col("salary_min") + F.col("salary_max")) / F.lit(2),
        )
    )
    .withColumn("source_system", F.lit("remoteok"))
    .withColumn("silver_processed_at", F.current_timestamp())
    .drop("description_text_step_1", "description_text_step_2")
)

display(
    silver_base_df.select(
        "source_job_id",
        "company",
        "position",
        "location",
        "salary_min",
        "salary_max",
        "has_salary",
        "is_remote",
        "tags_text",
    ).limit(20)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Deduplicar vagas
# MAGIC
# MAGIC Mantemos o registro mais recente por `source_job_id`.

# COMMAND ----------

dedup_window = Window.partitionBy("source_job_id").orderBy(F.col("ingested_at").desc())

silver_dedup_df = (
    silver_base_df
    .where(F.col("source_job_id").isNotNull())
    .withColumn("row_number", F.row_number().over(dedup_window))
    .where(F.col("row_number") == 1)
    .drop("row_number")
)

print(f"Registros antes da deduplicação: {silver_base_df.count()}")
print(f"Registros depois da deduplicação: {silver_dedup_df.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Selecionar schema final da Silver

# COMMAND ----------

silver_final_df = (
    silver_dedup_df
    .select(
        F.col("source_system"),
        F.col("source_job_id"),
        F.col("slug"),
        F.col("company"),
        F.col("company_normalized"),
        F.col("position"),
        F.col("position_normalized"),
        F.col("location"),
        F.col("location_normalized"),
        F.col("published_at_raw"),
        F.col("tags_array"),
        F.col("tags_text"),
        F.col("description_html"),
        F.col("description_text"),
        F.col("search_text"),
        F.col("salary_min"),
        F.col("salary_max"),
        F.col("salary_midpoint"),
        F.col("has_salary"),
        F.col("is_remote"),
        F.col("apply_url"),
        F.col("job_url"),
        F.col("payload_json"),
        F.col("run_id"),
        F.col("ingested_at"),
        F.col("file_path"),
        F.col("silver_processed_at"),
    )
)

display(silver_final_df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Gravar tabela Silver

# COMMAND ----------

(
    silver_final_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"OK: tabela Silver gravada em {SILVER_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Validações da Silver

# COMMAND ----------

silver_df = spark.table(SILVER_TABLE)

summary_df = silver_df.agg(
    F.count("*").alias("records"),
    F.countDistinct("source_job_id").alias("distinct_jobs"),
    F.countDistinct("company").alias("distinct_companies"),
    F.sum(F.when(F.col("description_text") != "", 1).otherwise(0)).alias("records_with_description"),
    F.sum(F.when(F.col("has_salary") == True, 1).otherwise(0)).alias("records_with_salary"),
    F.sum(F.when(F.col("is_remote") == True, 1).otherwise(0)).alias("records_marked_remote"),
)

display(summary_df)

quality_df = silver_df.select(
    F.sum(F.when(F.col("source_job_id").isNull(), 1).otherwise(0)).alias("missing_source_job_id"),
    F.sum(F.when(F.col("company").isNull() | (F.col("company") == ""), 1).otherwise(0)).alias("missing_company"),
    F.sum(F.when(F.col("position").isNull() | (F.col("position") == ""), 1).otherwise(0)).alias("missing_position"),
    F.sum(F.when(F.col("description_text").isNull() | (F.col("description_text") == ""), 1).otherwise(0)).alias("missing_description"),
    F.sum(F.when(F.col("salary_min") > F.col("salary_max"), 1).otherwise(0)).alias("invalid_salary_range"),
)

display(quality_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Exemplos para revisão

# COMMAND ----------

display(
    silver_df.select(
        "source_job_id",
        "company",
        "position",
        "location",
        "tags_text",
        "salary_min",
        "salary_max",
        "is_remote",
        F.substring("description_text", 1, 700).alias("description_preview"),
    )
    .orderBy(F.col("ingested_at").desc())
    .limit(10)
)

# COMMAND ----------

print()
print("=" * 70)
print("RESULTADO: TRANSFORMAÇÃO SILVER CONCLUÍDA")
print("=" * 70)
print(f"silver table: {SILVER_TABLE}")
print(f"records: {silver_df.count()}")
print("=" * 70)